# Word2Vec Evaluation Benchmark

Comprehensive evaluation of optimized vs default Skip-Gram embeddings.

- **Word Similarity:** WordSim-353, SimLex-999, MEN-3000 (Spearman ρ)
- **Word Analogies:** Google analogy dataset (~19k questions, 14 categories)
- **Comparison:** Optimized hyperparameters vs defaults

In [1]:
import csv
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

import joblib
import numpy as np
from scipy.stats import spearmanr

from word2vec.model import SkipGramNS
from word2vec.preprocessing import (
    apply_subsampling,
    build_negative_sampling_table,
    build_vocab,
    generate_training_pairs,
    subsample_probs,
    tokenize,
)
from word2vec.training import train

## 1. Load Optimized Model & Data

In [2]:
DATA_DIR = Path("../data")

# Optimized model
opt_model = SkipGramNS.load(DATA_DIR / "best_model.npz")
opt_vocab = joblib.load(DATA_DIR / "best_vocab.pkl")

print(f"Optimized model: vocab={opt_model.vocab_size:,}, dim={opt_model.embedding_dim}")

# Precompute normalized embeddings
def normalize(w_in):
    norms = np.linalg.norm(w_in, axis=1, keepdims=True) + 1e-10
    return w_in / norms

opt_norm = normalize(opt_model.w_in)

Optimized model: vocab=44,611, dim=300


## 2. Train Default Model

Train a model with the default hyperparameters from `__main__.py` for comparison.

In [3]:
# Load text8 tokens (shared between both models)
text = (DATA_DIR / "text8").read_text()
tokens = tokenize(text)

# Default hyperparameters (from __main__.py)
DEFAULT = dict(
    embedding_dim=100,
    window=5,
    neg_samples=5,
    lr_start=0.025,
    lr_min=1e-4,
    min_count=5,
    epochs=5,
    subsample_t=1e-5,
)

def_vocab = build_vocab(tokens, min_count=DEFAULT["min_count"])
rng = np.random.default_rng(42)
token_ids = np.array([def_vocab.word2id[w] for w in tokens if w in def_vocab.word2id])
discard_probs = subsample_probs(def_vocab.freqs, t=DEFAULT["subsample_t"])
token_ids = apply_subsampling(token_ids, discard_probs, rng)
pairs = generate_training_pairs(token_ids, window_size=DEFAULT["window"])
neg_table = build_negative_sampling_table(def_vocab.freqs)

def_model = SkipGramNS(vocab_size=len(def_vocab.words), embedding_dim=DEFAULT["embedding_dim"], seed=42)

print(f"Default model: vocab={def_model.vocab_size:,}, dim={def_model.embedding_dim}")
print(f"Training pairs: {len(pairs):,}")

train(
    def_model, pairs, neg_table,
    epochs=DEFAULT["epochs"],
    lr_start=DEFAULT["lr_start"],
    lr_min=DEFAULT["lr_min"],
    neg_samples=DEFAULT["neg_samples"],
)

def_norm = normalize(def_model.w_in)
print("\nDone.")

Default model: vocab=71,290, dim=100
Training pairs: 46,703,940


Epoch 1/5 | LR: 0.025000 | Avg Loss: 2.5511


Epoch 2/5 | LR: 0.018775 | Avg Loss: 2.2599


Epoch 3/5 | LR: 0.012550 | Avg Loss: 2.1737


Epoch 4/5 | LR: 0.006325 | Avg Loss: 2.1254


Epoch 5/5 | LR: 0.000100 | Avg Loss: 2.0993

Done.


## 3. Download Benchmarks

In [4]:
# --- WordSim-353 (already downloaded in HPO notebook) ---
wordsim_file = DATA_DIR / "wordsim353.csv"
if not wordsim_file.exists():
    ws_zip = DATA_DIR / "wordsim353.zip"
    urlretrieve("https://gabrilovich.com/resources/data/wordsim353/wordsim353.zip", str(ws_zip))
    with zipfile.ZipFile(ws_zip) as zf:
        with zf.open("combined.csv") as src, open(wordsim_file, "wb") as dst:
            dst.write(src.read())

# --- SimLex-999 ---
simlex_file = DATA_DIR / "SimLex-999.txt"
if not simlex_file.exists():
    sl_zip = DATA_DIR / "simlex999.zip"
    print("Downloading SimLex-999...")
    urlretrieve("https://fh295.github.io/SimLex-999.zip", str(sl_zip))
    with zipfile.ZipFile(sl_zip) as zf:
        with zf.open("SimLex-999/SimLex-999.txt") as src, open(simlex_file, "wb") as dst:
            dst.write(src.read())

# --- MEN-3000 ---
men_file = DATA_DIR / "MEN_natural_form_full.txt"
if not men_file.exists():
    men_zip = DATA_DIR / "MEN.zip"
    print("Downloading MEN-3000...")
    urlretrieve("https://staff.fnwi.uva.nl/e.bruni/resources/MEN.zip", str(men_zip))
    with zipfile.ZipFile(men_zip) as zf:
        with zf.open("MEN/MEN_dataset_natural_form_full") as src, open(men_file, "wb") as dst:
            dst.write(src.read())

# --- Google Analogy ---
analogy_file = DATA_DIR / "questions-words.txt"
if not analogy_file.exists():
    print("Downloading Google analogy dataset...")
    urlretrieve("http://download.tensorflow.org/data/questions-words.txt", str(analogy_file))

print("All benchmarks ready.")

All benchmarks ready.


## 4. Evaluation Helpers

In [5]:
def load_wordsim353():
    pairs = []
    with open(wordsim_file) as f:
        reader = csv.DictReader(f)
        for row in reader:
            pairs.append((row["Word 1"].lower(), row["Word 2"].lower(), float(row["Human (mean)"])))
    return pairs

def load_simlex999():
    pairs = []
    with open(simlex_file) as f:
        next(f)  # skip header
        for line in f:
            parts = line.strip().split("\t")
            pairs.append((parts[0].lower(), parts[1].lower(), float(parts[3])))
    return pairs

def load_men3000():
    pairs = []
    with open(men_file) as f:
        for line in f:
            parts = line.strip().split()
            pairs.append((parts[0].lower(), parts[1].lower(), float(parts[2])))
    return pairs

def eval_similarity(pairs, word2id, norm_embeddings):
    """Evaluate Spearman rho on word similarity pairs. Returns (rho, coverage_count, total_count)."""
    human, model_sims = [], []
    for w1, w2, score in pairs:
        if w1 in word2id and w2 in word2id:
            cos = float(norm_embeddings[word2id[w1]] @ norm_embeddings[word2id[w2]])
            model_sims.append(cos)
            human.append(score)
    if len(human) < 10:
        return 0.0, len(human), len(pairs)
    rho, _ = spearmanr(human, model_sims)
    return float(rho), len(human), len(pairs)

def load_analogies():
    """Load Google analogy dataset. Returns dict: category -> list of (a, b, c, d) tuples."""
    categories = {}
    current = None
    with open(analogy_file) as f:
        for line in f:
            line = line.strip()
            if line.startswith(":"):
                current = line[2:]
                categories[current] = []
            else:
                parts = line.lower().split()
                if len(parts) == 4:
                    categories[current].append(tuple(parts))
    return categories

def eval_analogies(categories, word2id, norm_embeddings):
    """Evaluate analogy accuracy per category. Returns dict: category -> (correct, total, accuracy)."""
    results = {}
    for cat, quads in categories.items():
        correct, total = 0, 0
        for a, b, c, d in quads:
            if not all(w in word2id for w in (a, b, c, d)):
                continue
            total += 1
            vec = norm_embeddings[word2id[b]] - norm_embeddings[word2id[a]] + norm_embeddings[word2id[c]]
            vec = vec / (np.linalg.norm(vec) + 1e-10)
            sims = norm_embeddings @ vec
            for w in (a, b, c):
                sims[word2id[w]] = -1
            if word2id[d] == np.argmax(sims):
                correct += 1
        acc = correct / total if total > 0 else 0.0
        results[cat] = (correct, total, acc)
    return results

print("Helpers defined.")

Helpers defined.


## 5. Word Similarity Benchmarks

Spearman ρ on three standard word similarity datasets.

In [6]:
ws353 = load_wordsim353()
sl999 = load_simlex999()
men3k = load_men3000()

sim_benchmarks = {"WordSim-353": ws353, "SimLex-999": sl999, "MEN-3000": men3k}

print(f"{'Benchmark':<15} {'Optimized ρ':>12} {'Default ρ':>12} {'Coverage (opt)':>16} {'Coverage (def)':>16}")
print("-" * 75)

sim_results = {}
for name, pairs in sim_benchmarks.items():
    rho_opt, cov_opt, total = eval_similarity(pairs, opt_vocab.word2id, opt_norm)
    rho_def, cov_def, _     = eval_similarity(pairs, def_vocab.word2id, def_norm)
    sim_results[name] = (rho_opt, rho_def, cov_opt, cov_def, total)
    print(f"{name:<15} {rho_opt:>12.4f} {rho_def:>12.4f} {cov_opt:>10}/{total:<5} {cov_def:>10}/{total:<5}")

Benchmark        Optimized ρ    Default ρ   Coverage (opt)   Coverage (def)
---------------------------------------------------------------------------
WordSim-353           0.7005       0.6768        350/353          351/353  
SimLex-999            0.2937       0.2861        980/999          992/999  
MEN-3000              0.6748       0.6067       2881/3000        2998/3000 


## 6. Google Analogy Test

Accuracy on ~19k analogy questions across 14 categories (semantic + syntactic).

In [7]:
analogy_cats = load_analogies()

print(f"{'Category':<30} {'Optimized':>12} {'Default':>12} {'Covered (opt)':>14}")
print("-" * 72)

opt_analogy = eval_analogies(analogy_cats, opt_vocab.word2id, opt_norm)
def_analogy = eval_analogies(analogy_cats, def_vocab.word2id, def_norm)

# Semantic vs syntactic split (first 5 categories are semantic in Google dataset)
SEMANTIC_CATS = {"capital-common-countries", "capital-world", "currency", "city-in-state", "family"}

sem_correct_opt, sem_total_opt = 0, 0
sem_correct_def, sem_total_def = 0, 0
syn_correct_opt, syn_total_opt = 0, 0
syn_correct_def, syn_total_def = 0, 0

for cat in analogy_cats:
    c_opt, t_opt, a_opt = opt_analogy[cat]
    c_def, t_def, a_def = def_analogy[cat]
    print(f"{cat:<30} {a_opt:>11.1%} {a_def:>11.1%} {t_opt:>8}/{len(analogy_cats[cat]):<5}")

    if cat in SEMANTIC_CATS:
        sem_correct_opt += c_opt; sem_total_opt += t_opt
        sem_correct_def += c_def; sem_total_def += t_def
    else:
        syn_correct_opt += c_opt; syn_total_opt += t_opt
        syn_correct_def += c_def; syn_total_def += t_def

total_opt = sem_correct_opt + syn_correct_opt
total_all_opt = sem_total_opt + syn_total_opt
total_def = sem_correct_def + syn_correct_def
total_all_def = sem_total_def + syn_total_def

print("-" * 72)
sem_acc_opt = sem_correct_opt / sem_total_opt if sem_total_opt else 0
sem_acc_def = sem_correct_def / sem_total_def if sem_total_def else 0
syn_acc_opt = syn_correct_opt / syn_total_opt if syn_total_opt else 0
syn_acc_def = syn_correct_def / syn_total_def if syn_total_def else 0
tot_acc_opt = total_opt / total_all_opt if total_all_opt else 0
tot_acc_def = total_def / total_all_def if total_all_def else 0

print(f"{'SEMANTIC':<30} {sem_acc_opt:>11.1%} {sem_acc_def:>11.1%}")
print(f"{'SYNTACTIC':<30} {syn_acc_opt:>11.1%} {syn_acc_def:>11.1%}")
print(f"{'TOTAL':<30} {tot_acc_opt:>11.1%} {tot_acc_def:>11.1%}")

Category                          Optimized      Default  Covered (opt)
------------------------------------------------------------------------
capital-common-countries             60.7%       42.9%      506/506  
capital-world                        37.9%       18.3%     2591/4524 
currency                             18.8%        9.1%      458/866  
city-in-state                        32.4%        8.4%     2060/2467 
family                               31.9%       18.3%      342/506  
gram1-adjective-to-adverb             4.8%        3.6%      870/992  
gram2-opposite                        5.3%        3.7%      506/812  
gram3-comparative                    24.5%       11.9%     1332/1332 
gram4-superlative                     8.7%        3.4%      756/1122 
gram5-present-participle             10.8%        5.8%     1056/1056 
gram6-nationality-adjective          75.9%       63.1%     1521/1599 
gram7-past-tense                     13.7%        5.6%     1482/1560 
gram8-plural   

## 7. Summary

In [8]:
import pandas as pd

rows = []
for name, (rho_opt, rho_def, cov_opt, cov_def, total) in sim_results.items():
    rows.append({
        "Benchmark": name,
        "Metric": "Spearman ρ",
        "Optimized": f"{rho_opt:.4f}",
        "Default": f"{rho_def:.4f}",
        "Δ": f"{rho_opt - rho_def:+.4f}",
        "Coverage (opt)": f"{cov_opt}/{total}",
    })

rows.append({
    "Benchmark": "Google Analogy (semantic)",
    "Metric": "Accuracy",
    "Optimized": f"{sem_acc_opt:.1%}",
    "Default": f"{sem_acc_def:.1%}",
    "Δ": f"{sem_acc_opt - sem_acc_def:+.1%}",
    "Coverage (opt)": f"{sem_total_opt}",
})
rows.append({
    "Benchmark": "Google Analogy (syntactic)",
    "Metric": "Accuracy",
    "Optimized": f"{syn_acc_opt:.1%}",
    "Default": f"{syn_acc_def:.1%}",
    "Δ": f"{syn_acc_opt - syn_acc_def:+.1%}",
    "Coverage (opt)": f"{syn_total_opt}",
})
rows.append({
    "Benchmark": "Google Analogy (total)",
    "Metric": "Accuracy",
    "Optimized": f"{tot_acc_opt:.1%}",
    "Default": f"{tot_acc_def:.1%}",
    "Δ": f"{tot_acc_opt - tot_acc_def:+.1%}",
    "Coverage (opt)": f"{total_all_opt}",
})

df = pd.DataFrame(rows)
print("Default params:  dim=100, window=5, neg=5, lr=0.025, min_count=5, epochs=5, t=1e-5")
print("Optimized params: dim=300, window=6, neg=13, lr≈0.016, min_count=11, epochs=10, t≈2.9e-5")
print()
df

Default params:  dim=100, window=5, neg=5, lr=0.025, min_count=5, epochs=5, t=1e-5
Optimized params: dim=300, window=6, neg=13, lr≈0.016, min_count=11, epochs=10, t≈2.9e-5



,Benchmark,Metric,Optimized,Default,Δ,Coverage (opt)
0,WordSim-353,Spearman ρ,0.7005,0.6768,+0.0237,350/353
1,SimLex-999,Spearman ρ,0.2937,0.2861,+0.0076,980/999
2,MEN-3000,Spearman ρ,0.6748,0.6067,+0.0682,2881/3000
3,Google Analogy (semantic),Accuracy,36.1%,16.1%,+20.0%,5957
4,Google Analogy (syntactic),Accuracy,26.4%,17.0%,+9.4%,9525
5,Google Analogy (total),Accuracy,30.1%,16.7%,+13.5%,15482
